# The U-Net and the diffusion schedule

The two pieces a diffusion model is made of, built and inspected before anything is trained.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 17 — Image Generation](../../../course-web-slides/ch17/index.html) &nbsp;·&nbsp; **Section:** 02 — Diffusion models

---

## The idea, stated once

An autoencoder can remove a **small** amount of noise. Repeat it in a loop and it can remove a **large** amount. Could it denoise an image made of *pure* noise?

Yes — and doing so hallucinates a new image out of nothing. These should more accurately be called **reverse** diffusion models; *diffusion* is the forward process of adding noise until the image disperses.

## The diffusion schedule

In [ ]:
import keras
from keras import ops
import numpy as np
import matplotlib.pyplot as plt

def diffusion_schedule(diffusion_times, min_signal_rate=0.02,
                       max_signal_rate=0.95):
    start_angle = ops.cast(ops.arccos(max_signal_rate), "float32")
    end_angle = ops.cast(ops.arccos(min_signal_rate), "float32")
    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)
    signal_rates = ops.cos(diffusion_angles)
    noise_rates = ops.sin(diffusion_angles)
    return noise_rates, signal_rates

t = ops.arange(0.0, 1.0, 0.01)
nr, sr = diffusion_schedule(t)
t = ops.convert_to_numpy(t)
nr = ops.convert_to_numpy(nr); sr = ops.convert_to_numpy(sr)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(t, nr, lw=2, label="noise rate")
a1.plot(t, sr, lw=2, label="signal rate")
a1.set_xlabel("diffusion time"); a1.legend(); a1.set_title("Cosine schedule")
a2.plot(nr, sr, lw=2)
a2.set_xlabel("noise rate"); a2.set_ylabel("signal rate")
a2.set_aspect("equal"); a2.set_title("noise^2 + signal^2 = 1")
plt.tight_layout(); plt.show()

print("identity holds:", np.allclose(nr**2 + sr**2, 1.0))

**Diffusion time runs from 1 to 0**: 1 is maximal noise, 0 is almost all signal. The cosine choice maintains `noise² + signal² = 1`, so total energy is constant as the mix shifts.

The two bounds keep the process away from its extremes — never quite pure signal (0.95), never quite pure noise (0.02). Both endpoints are numerically awkward and neither is needed.

## Seeing it on a real image

In [ ]:
from keras.datasets import cifar10
(x, _), _ = cifar10.load_data()
img = x[7].astype("float32") / 255
img = (img - img.mean()) / img.std()      # the noise has unit variance

rng = np.random.default_rng(0)
noise = rng.normal(size=img.shape).astype("float32")

times = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(times), figsize=(15, 3.2))
for ax, tt in zip(axes, times):
    n_, s_ = diffusion_schedule(ops.array([[[[tt]]]]))
    # .item(): NumPy 2 refuses float() on a non-0-d array, and these are shape (1,1,1,1).
    n_ = ops.convert_to_numpy(n_).item(); s_ = ops.convert_to_numpy(s_).item()
    mixed = s_ * img + n_ * noise
    show = np.clip((mixed - mixed.min()) / (mixed.max() - mixed.min()), 0, 1)
    ax.imshow(show); ax.axis("off")
    ax.set_title(f"t = {tt}\nsignal {s_:.2f}", fontsize=9)
plt.suptitle("The forward process — this is what the model learns to undo", y=1.06)
plt.tight_layout(); plt.show()

## The residual block

In [ ]:
from keras import layers

def residual_block(x, width):
    input_width = x.shape[3]
    if input_width == width:
        residual = x
    else:
        residual = layers.Conv2D(width, 1)(x)
    x = layers.BatchNormalization(center=False, scale=False)(x)
    x = layers.Conv2D(width, 3, padding="same", activation="swish")(x)
    x = layers.Conv2D(width, 3, padding="same")(x)
    return x + residual

Chapter 9's pattern, with `swish` instead of `relu` and normalization that learns **neither** a scale nor a centre — the residual path carries those.

## The U-Net

In [ ]:
def get_model(image_size, widths, block_depth):
    noisy_images = keras.Input(shape=(image_size, image_size, 3))
    noise_rates = keras.Input(shape=(1, 1, 1))

    x = layers.Conv2D(widths[0], 1)(noisy_images)
    n = layers.UpSampling2D(image_size, interpolation="nearest")(noise_rates)
    x = layers.Concatenate()([x, n])

    skips = []
    for width in widths[:-1]:
        for _ in range(block_depth):
            x = residual_block(x, width)
            skips.append(x)
        x = layers.AveragePooling2D(pool_size=2)(x)

    for _ in range(block_depth):
        x = residual_block(x, widths[-1])

    for width in reversed(widths[:-1]):
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        for _ in range(block_depth):
            x = layers.Concatenate()([x, skips.pop()])
            x = residual_block(x, width)

    pred_noise_masks = layers.Conv2D(3, 1, kernel_initializer="zeros")(x)
    return keras.Model([noisy_images, noise_rates], pred_noise_masks)

unet = get_model(image_size=128, widths=[32, 64, 96, 128], block_depth=2)
print(f"{unet.count_params():,} parameters")
print("input shapes:", [tuple(i.shape) for i in unet.inputs])
print("output shape:", tuple(unet.output.shape))

Three details worth naming:

**The scalar noise rate is upsampled to full image size** and concatenated as a channel — the standard way to feed a scalar condition to a convolutional network.

**`skips.pop()`** pairs each upsampling block with its mirror on the way down. Last in, first out is exactly the pairing the architecture diagram shows.

**`kernel_initializer="zeros"`** on the last layer: the model predicts only zeros at initialization, so its default assumption before training is *no noise*.

## The output is a noise mask, not an image

In [ ]:
probe_img = np.zeros((1, 128, 128, 3), dtype="float32")
probe_rate = np.array([[[[0.5]]]], dtype="float32")
out = unet([probe_img, probe_rate])
print("output:", out.shape, " all zeros at init:",
      bool(np.abs(np.array(out)).max() < 1e-8))
print()
print("Predicting what to REMOVE is an easier target than predicting")
print("what remains -- and it is why the denoise() step in the next")
print("notebook is a subtraction rather than a reconstruction.")

## Why the widths grow as the maps shrink

In [ ]:
for l in unet.layers:
    if isinstance(l, (layers.AveragePooling2D, layers.UpSampling2D)):
        print(f"{l.__class__.__name__:18s} -> {tuple(l.output.shape[1:])}")

128 → 64 → 32 → 16, then back. **The same space-for-semantics trade as every ConvNet in this course**, with the skip connections carrying the spatial detail that the downsampling discarded.

---

## What to take away

- The cosine schedule keeps `noise² + signal² = 1` as the mix shifts from one to the other.
- The U-Net takes **two inputs** — the noisy image and the noise rate.
- Skip connections preserve the detail that downsampling loses.
- The model predicts the **noise mask**, which is an easier target than the clean image.